# TechCore · Avance 2 — Modelo relacional (esquema en estrella)

**Objetivo:** partir del dataset crudo `ventas.csv` (formato *wide*: 1 fila = 1 venta con hasta 3 productos), limpiarlo y normalizarlo en un **modelo en estrella** con 2 tablas de hechos y 5 dimensiones, validando integridad referencial.

> **Corrección v2.** El `ventasTransformed.csv` del Avance 1 tenía los importes y descuentos mal interpretados por la configuración regional (por ejemplo `16000000.0` se leía como `160000000`). Esto inflaba ×10 las ventas. Este notebook parte ahora del CSV crudo y reconstruye todo con una validación explícita: `Total = Σ subtotales × (1 − descuento)`.

| Tabla | Tipo | PK | FK |
|---|---|---|---|
| Facturas | Hecho | FacturaID | SucursalID, VendedorID, ClienteID |
| DetalleFacturas | Hecho | DetalleID | FacturaID, ProductoID |
| Productos · Clientes · Sucursales · Vendedores · Ciudades | Dimensión | *ID | — |

## 1. Carga del dataset crudo

In [1]:
import unicodedata
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

RAW = '../01_Avance1_Limpieza/ventas.csv'
raw = pd.read_csv(RAW, encoding='utf-8', low_memory=False)
raw = raw.rename(columns={'Cliente_Nombre': 'ClienteNombre', 'edadcliente': 'EdadCliente',
                          'Direccion_Cliente': 'DireccionCliente'})
print(f'Filas crudas: {len(raw):,} | columnas: {raw.shape[1]}')
raw.head(3)

Filas crudas: 30,307 | columnas: 30


,VentaID,FechaVenta,HoraVenta,SucursalNombre,CiudadSucursal,VendedorNombre,ClienteNombre,GeneroCliente,EdadCliente,EmailCliente,TelefonoCliente,DireccionCliente,MetodoPago,NombreProducto1,MarcaProducto1,CantidadProducto1,PrecioUnitarioProducto1,SubtotalProducto1,NombreProducto2,MarcaProducto2,CantidadProducto2,PrecioUnitarioProducto2,SubtotalProducto2,NombreProducto3,MarcaProducto3,CantidadProducto3,PrecioUnitarioProducto3,SubtotalProducto3,DescuentoVenta,TotalVenta
0,1,2015-12-31,05:42:43,TechCore pereira,Pereíra,Amílcar Ortega-Alberto,Bienvenida Nebot-Fiol,F,37,bienvenida19@hotmail.com,+34806548767,cll 52 #32-98,Tarjeta Crédito,Apple MacBook Pro 16,Aple,1,9600000,9600000,MSI GS66 Stealth,MSI,2.0,8000000.0,16000000.0,Dell Latitude 7420,Del,1.0,5600000.0,5600000.0,0.0,31200000.0
1,2,2019-03-23,19:03:21,TechCore medellín #1,Medellin,Ana Sofía Llopis Blázquez,Teófila Bueno-Novoa,F,43,teófila35@gmail.com,+34 947255990,cll 96 #81-94,Billetera Digital,Acer Nitro 5,Acr,1,4000000,4000000,Lenovo ThinkPad X1 Carbon,Lenovo,1.0,6800000.0,6800000.0,NaN,NaN,NaN,NaN,NaN,0.0,10800000.0
2,3,2018-12-23,02:32:34,Techcore medellín #2,medellín,Juan José Porcel-Riera,Gilberto Chamorro Catalá,M,38,gilberto57@hotmail.com,+34 978 810 249,cra 28 #79-85,Tarjeta Débito,Dell Latitude 7420,Dell,2,5600000,11200000,Lenovo Legion 5 Pro,Lenvo,1.0,7200000.0,7200000.0,HP Pavilion 15,HP,1.0,3500000.0,3500000.0,NaN,21900000.0


## 2. Limpieza y estandarización

In [2]:
def sin_tildes(s):
    return ''.join(c for c in unicodedata.normalize('NFD', s) if unicodedata.category(c) != 'Mn')

def clave(s):
    return ' '.join(sin_tildes(str(s)).lower().split())

# --- 2.1 Duplicados de VentaID (se conserva la primera ocurrencia)
print('VentaID duplicados:', raw['VentaID'].duplicated().sum())
df = raw.drop_duplicates('VentaID', keep='first').reset_index(drop=True)

# --- 2.2 Marcas: diccionario de errores de tipeo -> forma canónica
MARCAS = {'lenvo': 'Lenovo', 'lenovo': 'Lenovo', 'hpp': 'HP', 'hp': 'HP',
          'del': 'Dell', 'dell': 'Dell', 'aple': 'Apple', 'apple': 'Apple',
          'azus': 'Asus', 'asus': 'Asus', 'acr': 'Acer', 'acer': 'Acer',
          'samsng': 'Samsung', 'samsung': 'Samsung', 'microsofft': 'Microsoft',
          'microsoft': 'Microsoft', 'msy': 'MSI', 'msi': 'MSI',
          'razor': 'Razer', 'razer': 'Razer'}
for i in (1, 2, 3):
    df[f'MarcaProducto{i}'] = df[f'MarcaProducto{i}'].map(lambda m: MARCAS[clave(m)] if pd.notna(m) else m)

# --- 2.3 Ciudad y sucursal: nombre canónico (con tildes) a partir de una clave sin tildes
CIUDADES = {'bogota': 'Bogotá', 'medellin': 'Medellín', 'cali': 'Cali', 'pereira': 'Pereira'}

def ciudad_canonica(row):
    for fuente in (row['CiudadSucursal'], row['SucursalNombre']):
        if pd.isna(fuente):
            continue
        k = clave(fuente)
        for ck, nombre in CIUDADES.items():
            if ck in k:
                return nombre
    return 'Sin ciudad'

def sucursal_canonica(nombre):
    k = clave(nombre).replace('techcore', '').replace('#', ' #').split()
    ciudad = CIUDADES[next(c for c in CIUDADES if c in k[0])]
    sufijo = ' '.join(k[1:]).replace('# ', '#')
    return f'TechCore {ciudad}' + (f' {sufijo}' if sufijo else '')

df['Ciudad'] = df.apply(ciudad_canonica, axis=1)
df['Sucursal'] = df['SucursalNombre'].map(sucursal_canonica)

# --- 2.4 Textos / medios de pago / descuento / fecha
df['MetodoPago'] = df['MetodoPago'].fillna('No especificado')
df['DescuentoVenta'] = df['DescuentoVenta'].fillna(0.0).astype(float)   # fracción: 0, .05, .10, .15
df['Fecha'] = pd.to_datetime(df['FechaVenta'])
df['Hora'] = df['HoraVenta']
df['VendedorNombre'] = df['VendedorNombre'].str.strip()

# --- 2.5 Importes numéricos
for i in (1, 2, 3):
    for c in ('Cantidad', 'PrecioUnitario', 'Subtotal'):
        df[f'{c}Producto{i}'] = pd.to_numeric(df[f'{c}Producto{i}'], errors='coerce').fillna(0)

print(f'Ventas únicas: {len(df):,}')
print('Marcas :', sorted(set(pd.concat([df[f"MarcaProducto{i}"] for i in (1,2,3)]).dropna())))
print('Ciudades:', sorted(df['Ciudad'].unique()))
print('Sucursales:', sorted(df['Sucursal'].unique()))

VentaID duplicados: 307


Ventas únicas: 30,000
Marcas : ['Acer', 'Apple', 'Asus', 'Dell', 'HP', 'Lenovo', 'MSI', 'Microsoft', 'Razer', 'Samsung']
Ciudades: ['Bogotá', 'Cali', 'Medellín', 'Pereira']
Sucursales: ['TechCore Bogotá #1', 'TechCore Bogotá #2', 'TechCore Cali', 'TechCore Medellín #1', 'TechCore Medellín #2', 'TechCore Pereira']


### 2.6 Validación contable: `Total = Σ subtotales × (1 − descuento)`

In [3]:
subtotal = sum(df[f'SubtotalProducto{i}'] for i in (1, 2, 3))
cant_x_precio = sum(df[f'CantidadProducto{i}'] * df[f'PrecioUnitarioProducto{i}'] for i in (1, 2, 3))
total_calc = subtotal * (1 - df['DescuentoVenta'])

assert (subtotal == cant_x_precio).all(), 'Subtotal != Cantidad x Precio'
assert ((df['TotalVenta'] - total_calc).abs() < 1).all(), 'Total no coincide con la fórmula'

df['SubtotalVenta'] = subtotal
print(f'Ventas brutas : ${subtotal.sum():>20,.0f}')
print(f'Descuentos    : ${(subtotal - df["TotalVenta"]).sum():>20,.0f}')
print(f'Ventas netas  : ${df["TotalVenta"].sum():>20,.0f}')
print('OK: subtotales y totales consistentes en las', len(df), 'ventas')

Ventas brutas : $     468,029,900,000
Descuentos    : $      11,538,230,000
Ventas netas  : $     456,491,670,000
OK: subtotales y totales consistentes en las 30000 ventas


## 3. Dimensiones

In [4]:
# Ciudades
ciudades = (df[['Ciudad']].drop_duplicates().sort_values('Ciudad').reset_index(drop=True)
            .rename(columns={'Ciudad': 'NombreCiudad'}))
ciudades.insert(0, 'CiudadID', ciudades.index + 1)

# Sucursales
sucursales = (df[['Sucursal', 'Ciudad']].drop_duplicates('Sucursal').sort_values('Sucursal').reset_index(drop=True))
sucursales.insert(0, 'SucursalID', sucursales.index + 1)
sucursales['CiudadID'] = sucursales['Ciudad'].map(dict(zip(ciudades.NombreCiudad, ciudades.CiudadID)))
sucursales = sucursales.rename(columns={'Sucursal': 'NombreSucursal'})[['SucursalID', 'NombreSucursal', 'CiudadID']]
assert sucursales['NombreSucursal'].is_unique

# Vendedores
vendedores = (df[['VendedorNombre']].drop_duplicates().sort_values('VendedorNombre').reset_index(drop=True)
              .rename(columns={'VendedorNombre': 'NombreVendedor'}))
vendedores.insert(0, 'VendedorID', vendedores.index + 1)

# Clientes: la clave natural es el email; si falta, nombre + teléfono + dirección
df['ClienteKey'] = df['EmailCliente'].str.strip().str.lower()
sin_email = df['ClienteKey'].isna()
df.loc[sin_email, 'ClienteKey'] = ('sin-email|' + df['ClienteNombre'].str.lower().str.strip() + '|' +
                                   df['TelefonoCliente'].fillna('') + '|' + df['DireccionCliente'].fillna(''))

clientes = (df[['ClienteKey', 'ClienteNombre', 'GeneroCliente', 'EdadCliente', 'EmailCliente',
                'TelefonoCliente', 'DireccionCliente']]
            .drop_duplicates('ClienteKey').sort_values('ClienteKey').reset_index(drop=True))
clientes.insert(0, 'ClienteID', clientes.index + 1)
map_cliente = dict(zip(clientes.ClienteKey, clientes.ClienteID))
clientes = clientes.drop(columns='ClienteKey').rename(columns={
    'ClienteNombre': 'Nombre', 'GeneroCliente': 'Genero', 'EdadCliente': 'Edad',
    'EmailCliente': 'Email', 'TelefonoCliente': 'Telefono', 'DireccionCliente': 'Direccion'})

# Productos: formato wide -> long, una fila por (Nombre, Marca); precio = moda
largo = pd.concat([
    df[[f'NombreProducto{i}', f'MarcaProducto{i}', f'PrecioUnitarioProducto{i}']].set_axis(['Nombre', 'Marca', 'Precio'], axis=1)
    for i in (1, 2, 3)], ignore_index=True).dropna(subset=['Nombre'])
productos = (largo.groupby(['Nombre', 'Marca'], as_index=False)
             .agg(Precio=('Precio', lambda s: s.mode().iloc[0]))
             .sort_values(['Marca', 'Nombre']).reset_index(drop=True))
productos.insert(0, 'ProductoID', productos.index + 1)
assert productos['Nombre'].is_unique, 'Un producto aparece con dos marcas'

for nombre, t in [('Ciudades', ciudades), ('Sucursales', sucursales), ('Vendedores', vendedores),
                  ('Clientes', clientes), ('Productos', productos)]:
    print(f'{nombre:<11}: {len(t):>6,}')

Ciudades   :      4
Sucursales :      6
Vendedores :     30
Clientes   : 16,715
Productos  :     40


## 4. Tablas de hechos

In [5]:
map_suc = dict(zip(sucursales.NombreSucursal, sucursales.SucursalID))
map_ven = dict(zip(vendedores.NombreVendedor, vendedores.VendedorID))

facturas = pd.DataFrame({
    'FacturaID': df['VentaID'],
    'Fecha': df['Fecha'],
    'Hora': df['Hora'],
    'SucursalID': df['Sucursal'].map(map_suc),
    'VendedorID': df['VendedorNombre'].map(map_ven),
    'ClienteID': df['ClienteKey'].map(map_cliente),
    'MetodoPago': df['MetodoPago'],
    'Subtotal': df['SubtotalVenta'].astype('int64'),
    'DescuentoPct': df['DescuentoVenta'],
    'Total': df['TotalVenta'].round().astype('int64'),
})
facturas['MontoDescuento'] = facturas['Subtotal'] - facturas['Total']
assert facturas['FacturaID'].is_unique
assert facturas[['SucursalID', 'VendedorID', 'ClienteID']].notna().all().all()

map_prod = dict(zip(zip(productos.Nombre, productos.Marca), productos.ProductoID))
partes = []
for i in (1, 2, 3):
    p = df[['VentaID', f'NombreProducto{i}', f'MarcaProducto{i}', f'CantidadProducto{i}',
            f'PrecioUnitarioProducto{i}', f'SubtotalProducto{i}', 'DescuentoVenta']].copy()
    p.columns = ['FacturaID', 'Nombre', 'Marca', 'Cantidad', 'PrecioUnitario', 'Subtotal', 'DescuentoPct']
    partes.append(p)
detalle = pd.concat(partes, ignore_index=True)
detalle = detalle[detalle['Nombre'].notna() & (detalle['Cantidad'] > 0)]
detalle = detalle.sort_values('FacturaID', kind='stable').reset_index(drop=True)
detalle['ProductoID'] = [map_prod[k] for k in zip(detalle.Nombre, detalle.Marca)]
detalle['DetalleID'] = detalle.index + 1
detalle['SubtotalNeto'] = (detalle['Subtotal'] * (1 - detalle['DescuentoPct'])).round(2)
detalle = detalle[['DetalleID', 'FacturaID', 'ProductoID', 'Cantidad', 'PrecioUnitario', 'Subtotal',
                   'DescuentoPct', 'SubtotalNeto']]
detalle = detalle.astype({'Cantidad': 'int64', 'PrecioUnitario': 'int64', 'Subtotal': 'int64'})

# La suma del detalle neto debe cuadrar con la cabecera
diff = abs(detalle['SubtotalNeto'].sum() - facturas['Total'].sum())
print(f'Facturas: {len(facturas):,} | Detalle: {len(detalle):,} | diferencia detalle vs cabecera: ${diff:,.2f}')
assert diff < len(facturas), 'El detalle no cuadra con la cabecera'

Facturas: 30,000 | Detalle: 60,059 | diferencia detalle vs cabecera: $0.00


## 5. Integridad referencial

In [6]:
def huerfanos(hijo, fk, padre, pk):
    n = (~hijo[fk].isin(padre[pk])).sum()
    print(f"[{'OK' if n == 0 else 'ERROR'}] {fk:<11} -> {pk:<11}: {n} huérfanos")
    return n == 0

ok = all([
    huerfanos(detalle, 'FacturaID', facturas, 'FacturaID'),
    huerfanos(detalle, 'ProductoID', productos, 'ProductoID'),
    huerfanos(facturas, 'SucursalID', sucursales, 'SucursalID'),
    huerfanos(facturas, 'VendedorID', vendedores, 'VendedorID'),
    huerfanos(facturas, 'ClienteID', clientes, 'ClienteID'),
    huerfanos(sucursales, 'CiudadID', ciudades, 'CiudadID'),
])
assert ok, 'Hay claves huérfanas'
print('\nRESULTADO: modelo íntegro')

[OK] FacturaID   -> FacturaID  : 0 huérfanos
[OK] ProductoID  -> ProductoID : 0 huérfanos
[OK] SucursalID  -> SucursalID : 0 huérfanos
[OK] VendedorID  -> VendedorID : 0 huérfanos
[OK] ClienteID   -> ClienteID  : 0 huérfanos
[OK] CiudadID    -> CiudadID   : 0 huérfanos

RESULTADO: modelo íntegro


## 6. Verificación de negocio (cifras corregidas)

In [7]:
ventas_ciudad = (facturas.merge(sucursales, on='SucursalID').merge(ciudades, on='CiudadID')
                 .groupby('NombreCiudad').agg(Ventas=('Total', 'sum'), Facturas=('FacturaID', 'count'))
                 .sort_values('Ventas', ascending=False))
ventas_ciudad['%'] = (ventas_ciudad['Ventas'] / ventas_ciudad['Ventas'].sum() * 100).round(1)
ventas_ciudad

,Ventas,Facturas,%
NombreCiudad,,,
Medellín,203861345000,13460,44.7
Bogotá,139607510000,9107,30.6
Pereira,56619400000,3723,12.4
Cali,56403415000,3710,12.4


In [8]:
(detalle.merge(productos[['ProductoID', 'Marca']], on='ProductoID')
 .groupby('Marca').agg(VentasNetas=('SubtotalNeto', 'sum'), Unidades=('Cantidad', 'sum'))
 .sort_values('VentasNetas', ascending=False))

,VentasNetas,Unidades
Marca,,
Lenovo,1.149497e+11,21697
HP,9.440325e+10,19877
Dell,8.460436e+10,16189
Apple,7.607139e+10,10783
Asus,3.238433e+10,8097
Acer,2.145418e+10,7178
Samsung,1.245680e+10,2618
MSI,8.311020e+09,1374
Microsoft,8.017840e+09,1717


In [9]:
(detalle.merge(productos, on='ProductoID').groupby(['Nombre', 'Marca'], as_index=False)
 .agg(Unidades=('Cantidad', 'sum'), Ingreso=('SubtotalNeto', 'sum'))
 .sort_values('Unidades', ascending=False).head(10).reset_index(drop=True))

,Nombre,Marca,Unidades,Ingreso
0,HP Spectre x360,HP,7726,3.919370e+10
1,Lenovo Legion 5 Pro,Lenovo,5861,4.116420e+10
2,Lenovo ThinkPad X1 Carbon,Lenovo,5814,3.853458e+10
3,Lenovo Yoga 7i,Lenovo,5052,2.170498e+10
4,HP Omen 16,HP,5011,2.933220e+10
5,Lenovo IdeaPad 5,Lenovo,4970,1.354598e+10
6,Dell XPS 13,Dell,4526,2.119896e+10
7,Dell Latitude 7420,Dell,4507,2.462880e+10
8,HP Pavilion 15,HP,4029,1.376095e+10
9,Dell Alienware m15,Dell,3658,2.855200e+10


## 7. Exportación del modelo

In [10]:
tablas = {'Facturas': facturas, 'DetalleFacturas': detalle, 'Productos': productos,
          'Clientes': clientes, 'Sucursales': sucursales, 'Vendedores': vendedores, 'Ciudades': ciudades}

with pd.ExcelWriter('modeloVentas.xlsx', engine='openpyxl') as writer:
    for nombre, t in tablas.items():
        t.to_excel(writer, sheet_name=nombre, index=False)

print('modeloVentas.xlsx generado')
for nombre, t in tablas.items():
    print(f'  {nombre:<16}{len(t):>8,}')

modeloVentas.xlsx generado
  Facturas          30,000
  DetalleFacturas   60,059
  Productos             40
  Clientes          16,715
  Sucursales             6
  Vendedores            30
  Ciudades               4
